# 03 — Capacity: what binds first, and at what load

Everything after this notebook depends on one number: **how many concurrent
sessions this cluster holds while keeping its TTFT budget.** Pick it before
measuring and it is a wish; derive it here and 04 onwards have a defensible
operating point.

The second question matters as much. An admission policy reads **one**
signal. If that signal never moves, the policy never fires, and an A/B
against it compares two identical configurations. So before testing any
policy we establish **which resource actually runs out** — and whether a
threshold on it is reachable at all.

That question is AdmitPerf's first-class concern, so this notebook uses
`admitperf.Watch` to record the signals rather than hand-rolling it. Watch
polls the engine's own `/metrics` once a second; reading the same thing
through Prometheus gets you its 15-second scrape interval, and a 12-second
load burst can fall between two scrapes and leave no trace at all.

Run `make cluster` and `make ssh` first. This generates real load.


In [1]:
import json, subprocess, sys, threading, time
from pathlib import Path

ROOT = Path.cwd().parents[1] if Path.cwd().name == '03-capacity' else Path.cwd()
sys.path.insert(0, str(ROOT / 'benchmarking'))
%load_ext autoreload
%autoreload 2

from admitperf.report import Report as ApReport
from admitperf.watch import Watch

from prom import Prom
from utils import Report
from slos import SLOS, TTFT_BUDGET_S, budget_rows, set_target_sessions

prom   = Prom()
report = Report('03-capacity')
v = lambda q: (lambda r: float(r[0][1]) if r else 0.0)(prom.query(q))

#: The engine's own metrics, through the tunnel. :30801 is decode, :30800
#: prefill. Decode is the pool an admission policy protects.
DECODE_METRICS = 'http://127.0.0.1:30801/metrics'
TRACES = ROOT / 'benchmarking' / '03-capacity' / 'traces'
TRACES.mkdir(exist_ok=True)


## Objective


In [2]:
report.table(['stage of the speech path', 'budget', ''], budget_rows(),
             note='these are serial on the way to first audio, so the engine '
                  'gets a slice of the 1.5s a listener tolerates, not all of it.')

report.objective(
    'How many concurrent sessions hold the TTFT budget, and which resource '
    'runs out first — KV cache, decode slots, or scheduling?',
    slos=['ttft_p95', 'preemptions', 'shed_rate'],
    falsified_by=f'TTFT p95 crossing {TTFT_BUDGET_S:g}s at any load we intend '
                 'to serve, or no signal moving at all — which would mean we '
                 'never reached saturation and cannot claim a capacity limit.')


  stage of the speech path  budget                            
  ------------------------  -------  -------------------------
  ASR finalize              300 ms                            
  retrieval and tools       400 ms                            
  LLM TTFT                  500 ms   what 03 measures         
  TTS first chunk           300 ms                            
  total                     1500 ms  what a listener tolerates
  -- these are serial on the way to first audio, so the engine gets a slice of the 1.5s a listener tolerates, not all of it.

Objective
  How many concurrent sessions hold the TTFT budget, and which resource runs out first — KV cache, decode slots, or scheduling?
  SLO under test            target
  ------------------------  ------
  TTFT p95                  0.5s  
  preemptions               0     
  shed rate at target load  0.05  
  Falsified by: TTFT p95 crossing 0.5s at any load we intend to serve, or no signal moving at all — which would mean we 

Report(title='03-capacity', subtitle='', blocks=[_Block(kind='table', payload=(['stage of the speech path', 'budget', ''], [['ASR finalize', '300 ms', ''], ['retrieval and tools', '400 ms', ''], ['LLM TTFT', '500 ms', 'what 03 measures'], ['TTS first chunk', '300 ms', ''], ['total', '1500 ms', 'what a listener tolerates']]), note='these are serial on the way to first audio, so the engine gets a slice of the 1.5s a listener tolerates, not all of it.'), _Block(kind='heading', payload='Objective', note=''), _Block(kind='text', payload='How many concurrent sessions hold the TTFT budget, and which resource runs out first — KV cache, decode slots, or scheduling?', note=''), _Block(kind='table', payload=(['SLO under test', 'target'], [['TTFT p95', '0.5s'], ['preemptions', '0'], ['shed rate at target load', '0.05']]), note=''), _Block(kind='text', payload='Falsified by: TTFT p95 crossing 0.5s at any load we intend to serve, or no signal moving at all — which would mean we never reached saturat

## 1. What the hardware allows

Two independent ceilings, and the lower one is the real limit:

- **KV cache** — each slice holds a fixed number of token-KVs
- **decode slots** — `--max-num-seqs` caps concurrent sequences regardless of
  how much KV is free

Which binds depends entirely on prompt length, so it is computed from the
measured prompt rather than assumed. This is the arithmetic AdmitPerf's
README makes the case for: on one A100-40 a `kv_used_fraction` cannot exceed
~0.35, so a policy thresholded at 0.90 is unreachable at any arrival rate.


In [3]:
KIB_PER_TOKEN = 56      # 28 layers x 4 kv-heads x 128 head-dim x 2 (k+v) x 2 bytes
SLICE_KV_GIB  = 4.10    # measured on a slice, not divided down from the card
MAX_NUM_SEQS  = 64      # from the manifests

p_tok = v('sum(increase(vllm:prompt_tokens_total[1h]))')
p_req = v('sum(increase(vllm:request_success_total[1h]))')
PROMPT_TOKENS = p_tok / p_req if p_req else 0

kv_per_seq_gib = PROMPT_TOKENS * KIB_PER_TOKEN / 1024 / 1024
seqs_by_kv = SLICE_KV_GIB / kv_per_seq_gib if kv_per_seq_gib else float('inf')
binding    = 'KV cache' if seqs_by_kv < MAX_NUM_SEQS else 'decode slots'

report.heading('1. What the hardware allows')
report.table(['ceiling', 'concurrent sequences', 'why'], [
    ['KV cache', f'{seqs_by_kv:.1f}',
     f'{SLICE_KV_GIB} GiB / {kv_per_seq_gib:.2f} GiB per prompt'],
    ['decode slots', f'{MAX_NUM_SEQS}', '--max-num-seqs, independent of KV'],
], note=f'measured prompt length: {PROMPT_TOKENS:,.0f} tokens. '
        'The lower number is the ceiling that applies.')
report.conclusion(
    f'Prompts here average {PROMPT_TOKENS:,.0f} tokens, so each sequence costs '
    f'{kv_per_seq_gib:.2f} GiB of KV and a {SLICE_KV_GIB} GiB slice holds '
    f'{seqs_by_kv:.1f} of them — against {MAX_NUM_SEQS} decode slots. '
    f'{binding} binds first. This is a property of the workload, not the '
    'hardware: at short prompts the same cluster binds on slots instead, and '
    'a KV-threshold policy would never fire.')



1. What the hardware allows
  ceiling       concurrent sequences  why                              
  ------------  --------------------  ---------------------------------
  KV cache                      15.8  4.1 GiB / 0.26 GiB per prompt    
  decode slots                    64  --max-num-seqs, independent of KV
  -- measured prompt length: 4,868 tokens. The lower number is the ceiling that applies.

  >> Prompts here average 4,868 tokens, so each sequence costs 0.26 GiB of KV and a 4.1 GiB slice holds 15.8 of them — against 64 decode slots. KV cache binds first. This is a property of the workload, not the hardware: at short prompts the same cluster binds on slots instead, and a KV-threshold policy would never fire.



Report(title='03-capacity', subtitle='', blocks=[_Block(kind='table', payload=(['stage of the speech path', 'budget', ''], [['ASR finalize', '300 ms', ''], ['retrieval and tools', '400 ms', ''], ['LLM TTFT', '500 ms', 'what 03 measures'], ['TTS first chunk', '300 ms', ''], ['total', '1500 ms', 'what a listener tolerates']]), note='these are serial on the way to first audio, so the engine gets a slice of the 1.5s a listener tolerates, not all of it.'), _Block(kind='heading', payload='Objective', note=''), _Block(kind='text', payload='How many concurrent sessions hold the TTFT budget, and which resource runs out first — KV cache, decode slots, or scheduling?', note=''), _Block(kind='table', payload=(['SLO under test', 'target'], [['TTFT p95', '0.5s'], ['preemptions', '0'], ['shed rate at target load', '0.05']]), note=''), _Block(kind='text', payload='Falsified by: TTFT p95 crossing 0.5s at any load we intend to serve, or no signal moving at all — which would mean we never reached saturat

## 2. The sweep

Same workload, same seed, one variable: concurrent sessions. Each step runs
`load.py` while `admitperf.Watch` records **every** signal from decode's
`/metrics` — not just the one we expect to move. That is what lets the
finding say *"your signal never budged, but queue depth hit 61"* instead of
only confirming what we went looking for.

A failed scrape is logged as a failure, never as zeros — zeros would read as
an idle cluster and make the whole trace look like headroom.


In [ ]:
SESSIONS = [1, 2, 4, 8, 16, 24, 32, 48]
SEED     = 1
REPEAT   = 8    # passes per session, so each step lasts ~90s

def record_during(fn, trace_path):
    """Run fn while Watch samples decode's /metrics once a second."""
    watch = Watch(DECODE_METRICS, trace_path, interval=1.0)
    stop = threading.Event()

    def poll():
        while not stop.is_set():
            watch.once()
            stop.wait(1.0)

    t = threading.Thread(target=poll, daemon=True); t.start()
    try:
        return fn()
    finally:
        stop.set(); t.join(timeout=5); watch.log.close()


def run_load(n):
    """One sweep step. Reads load.py's bundle rather than its stdout."""
    run_id = f'capacity-s{n}'
    subprocess.run(
        [sys.executable, str(ROOT / 'benchmarking' / 'load.py'),
         '--sessions', str(n), '--seed', str(SEED), '--max-tokens', '200',
         '--tool-gap', '0.2', '--repeat', str(REPEAT), '--run', run_id],
        capture_output=True, text=True, timeout=1800, cwd=ROOT)
    res = json.loads((ROOT / 'infra' / 'metrics' / run_id / 'load.json')
                     .read_text())['results']
    lat = sorted(r['latency_s'] for r in res if r['status'] == 200)
    return {
        'ok': sum(1 for r in res if r['status'] == 200),
        # 503 is capacity, 429 is quota. Conflating them lets a rate-limited
        # sweep look like a saturated cluster.
        'shed_503': sum(1 for r in res if r['status'] == 503),
        'shed_429': sum(1 for r in res if r['status'] == 429),
        'errors':   sum(1 for r in res if r['status'] not in (200, 429, 503)),
        'client_p95': lat[min(len(lat) - 1, int(len(lat) * 0.95))] if lat else 0.0,
    }


# A restart invalidates every counter delta across it.
restarts_before = v('sum(kube_pod_container_status_restarts_total{pod=~"vllm-.*"})')

rows = []
for n in SESSIONS:
    trace = TRACES / f'capacity-s{n}.jsonl'
    client = record_during(lambda: run_load(n), trace)
    ap = ApReport.from_log(trace)

    sig = {name: (rng or {}) for name, rng in ap.signals().items()}
    rows.append({
        'sessions': n, 'trace': trace, 'ap': ap,
        'ttft_p95': prom.quantiles('ttft', qs=(0.95,), window='3m').get(0.95, 0.0),
        'kv_max':    sig.get('kv_pressure', {}).get('max', 0.0),
        'queue_max': sig.get('queue_depth', {}).get('max', 0.0),
        'run_max':   sig.get('running', {}).get('max', 0.0),
        'samples':   len(ap.states), 'scrape_errors': ap.scrape_errors,
        **client})
    print(f"  {n:>3} sessions  {client['ok']:>4} ok  {client['shed_503']:>3} 503  "
          f"{client['shed_429']:>3} 429  {client['errors']:>2} err  "
          f"client p95 {client['client_p95']:.2f}s  "
          f"KV max {rows[-1]['kv_max']:.1%}  queue max {rows[-1]['queue_max']:.0f}  "
          f"({rows[-1]['samples']} samples)")
    time.sleep(30)   # drain, so the next step starts clean

restarts_after = v('sum(kube_pod_container_status_restarts_total{pod=~"vllm-.*"})')


## 3. Could a policy have fired?

AdmitPerf's own finding, at the heaviest load. This is the question it exists
to answer first, and the one it reports that no surveyed paper does: the
observed **range** of the quantity a policy reads.


In [ ]:
report.heading('3. Could a policy have fired?')
heaviest = rows[-1]
print(heaviest['ap'].text())

xs = [r['sessions'] for r in rows]
report.line('Latency against load', xs,
            {'TTFT p95 (s)':  [r['ttft_p95'] for r in rows],
             'client p95 (s)': [r['client_p95'] for r in rows]},
            xlabel='concurrent sessions', ylabel='seconds',
            hlines={f'{TTFT_BUDGET_S:g}s engine budget': TTFT_BUDGET_S},
            note='TTFT is the engine\'s share of the speech budget; client p95 '
                 'is the whole turn and is always larger')

report.line('What ran out', xs,
            {'KV pressure (%)':  [r['kv_max'] * 100 for r in rows],
             'queue depth':      [r['queue_max'] for r in rows],
             'running':          [r['run_max'] for r in rows]},
            xlabel='concurrent sessions', ylabel='max during the step',
            hlines={'0.90 threshold': 90},
            note='a flat line means that resource was never scarce, so a policy '
                 'reading it could not have fired')

report.table(['sessions', 'ok', '503', '429', 'err', 'TTFT p95', 'client p95',
              'KV max', 'queue max', 'samples'],
             [[r['sessions'], r['ok'], r['shed_503'], r['shed_429'], r['errors'],
               f"{r['ttft_p95']:.3f}s", f"{r['client_p95']:.2f}s",
               f"{r['kv_max']:.1%}", f"{r['queue_max']:.0f}", r['samples']]
              for r in rows],
             note='503 is the cluster refusing on capacity; 429 is the tenant '
                  'quota and means that step measured the limiter, not the GPU.')


## 4. The derived target


In [ ]:
report.heading('4. The derived target')
target = SLOS['ttft_p95'].target

# Guards first. An earlier version of this notebook derived a target from a
# sweep where three steps served nothing and one reported a NEGATIVE request
# count — and printed PASS. A capacity number from a run that never loaded
# the cluster is worse than no number, because in a report they look alike.
no_traffic = [r['sessions'] for r in rows if r['ok'] == 0]
quota_hit  = [r['sessions'] for r in rows if r['shed_429'] > 0]
errored    = [r['sessions'] for r in rows if r['errors'] > 0]
blind      = [r['sessions'] for r in rows if r['scrape_errors'] > 0]
crashed    = restarts_after > restarts_before

report.verdict(not no_traffic, 'every load level served at least one request'
               + (f' (none at {no_traffic})' if no_traffic else ''))
report.verdict(not quota_hit,
               'no step hit the tenant quota — a 429 is client-attributable '
               'and says nothing about capacity'
               + (f' (429s at {quota_hit})' if quota_hit else ''))
report.verdict(not crashed, 'no engine restarted during the sweep'
               + (' — counters reset, so deltas across it are void' if crashed else ''))
report.verdict(not errored, 'no step produced transport errors'
               + (f' (errors at {errored})' if errored else ''))
report.verdict(not blind, 'every step scraped the engine cleanly'
               + (f' (scrape failures at {blind})' if blind else ''))

sound = not (no_traffic or quota_hit or crashed or errored or blind)
ok    = [r for r in rows if r['ok'] > 0 and 0 < r['ttft_p95'] <= target]
best  = max((r['sessions'] for r in ok), default=0)

kv_peak, q_peak = max(r['kv_max'] for r in rows), max(r['queue_max'] for r in rows)
kv_moved = kv_peak - min(r['kv_max'] for r in rows)
scarce = ('KV cache' if kv_moved > 0.05 else
          'queue depth' if q_peak >= 1 else 'nothing')
report.verdict(scarce != 'nothing', 'a scarcity signal moved during the sweep')

topped_out = best < max(SESSIONS)
report.verdict(topped_out,
               'the sweep found a limit rather than running out of test load')
if ok:
    report.slo('ttft_p95', max(r['ttft_p95'] for r in ok))

if sound and best:
    where = set_target_sessions(best, evidence=(
        f'03-capacity swept {SESSIONS} at seed {SEED}, repeat {REPEAT}; '
        f'TTFT p95 within {target}s through {best} sessions; '
        f'peak KV {kv_peak:.1%}, peak queue {q_peak:.0f}; '
        'signals recorded by admitperf.Watch off the engine at 1s'))
    print(f'  wrote {where.relative_to(ROOT)}')
    report.conclusion(
        f'TTFT p95 held within the {target:g}s engine budget through {best} '
        f'concurrent sessions, which is the operating point 04 onwards will use. '
        f'The signal that moved was {scarce}: KV pressure peaked at {kv_peak:.1%} '
        f'and the queue reached {q_peak:.0f}. '
        + (f'A kv_threshold at 0.90 could NOT have fired here — nothing above '
           f'{kv_peak:.0%} was ever observed — so 04 must either set its '
           f'threshold from this curve or drive the cluster harder. '
           if kv_peak < 0.90 else 'A 0.90 threshold is reachable, so 04 can use it. ')
        + ('' if topped_out else
           f'NOTE: {best} is the highest load TESTED and it met the budget '
           f'comfortably, so it is a lower bound, not the limit.'))
else:
    why = '; '.join(filter(None, [
        f'no traffic at {no_traffic}' if no_traffic else '',
        f'tenant quota refused requests at {quota_hit}' if quota_hit else '',
        'an engine restarted and reset its counters' if crashed else '',
        f'transport errors at {errored}' if errored else '',
        f'scrape failures at {blind}' if blind else '',
        f'no load level met the {target:g}s budget' if sound and not best else '']))
    report.conclusion(
        f'NO TARGET WRITTEN. {why}. Until that is fixed any capacity figure '
        'here measures something other than the cluster.')


## Verdict


In [ ]:
print()
fails = report.failures()
if fails:
    print('NOT SAFE TO PROCEED — fix these first:')
    for f in fails:
        print(f'  - {f}')
else:
    from slos import target_sessions
    print(f'Capacity is {target_sessions()} concurrent sessions. Next: 04-admission')
print()
print(f'Traces for 04 to replay: {TRACES.relative_to(ROOT)}/')
